# Ablation + Inference Benchmark for IJCIM Revision

Fills `[TODO]` cells in Tables 3 (ablation), 4 (threshold sweep), and 5 (inference latency).

**Run order (each cell is small and isolated to localise any kernel crash):**

| Cell | What it does | Time |
|---|---|---|
| 1a | Plain Python imports (no TF) | <1 s |
| 1b | `import tensorflow` ALONE — if the kernel dies, this is the culprit | <5 s |
| 1c | Config + path asserts | <1 s |
| 2 | Load all 6 CSVs (no TF) | ~30 s |
| 3 | Inference benchmark → `benchmark_results.json` | ~1 min |
| 4 | Ablation of 6 variants → `ablation_results.json` | 30–60 min (GPU) |
| 5 | Threshold sweep → `threshold_sweep.json` | ~1 min |

**If cell 1b kills the kernel**, restart and set `USE_GPU = False` in cell 1a. The benchmark still produces useful CPU numbers; the ablation will be slow but possible.

## Cell 1a — Plain Python (no TF)

In [ ]:
import os, sys, json, glob, time

USE_GPU = True   # <-- set False if cell 1b kills the kernel

if not USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')

import numpy as np
import pandas as pd

print('Cell 1a OK. Python', sys.version.split()[0])
print('cwd:', os.getcwd())

## Cell 1b — TensorFlow import (the suspect)

If this is the cell that dies, restart the kernel, then in cell 1a flip `USE_GPU = False` and rerun 1a → 1b → ...

In [ ]:
import tensorflow as tf

# Enable memory growth so TF doesn't pre-allocate all VRAM (common kernel-killer on shared hubs).
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print('  could not enable memory growth on', gpu, ':', e)

print('TF', tf.__version__)
print('Visible GPUs:', tf.config.list_physical_devices('GPU'))
print('Visible CPUs:', tf.config.list_physical_devices('CPU'))

## Cell 1c — Config & path asserts

In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

MODEL_PATH = '../models/best_cnnlstm_model_vel_v2.keras'
DATA_ROOT  = '../data'

SEQ_LEN = 5
GRID = 48
NUM_CHANNELS = 2
NUM_CLASSES = 6
X_MIN, X_MAX = -0.5, 0.5
Y_MIN, Y_MAX = 0.1, 1.0

N_WARMUP = 50
N_TRIALS = 1000
ABLATION_EPOCHS = 40
RANDOM_SEED = 42

print('MODEL_PATH exists:', os.path.exists(MODEL_PATH))
print('DATA_ROOT  exists:', os.path.isdir(DATA_ROOT))
assert os.path.exists(MODEL_PATH), f'missing {MODEL_PATH}'
assert os.path.isdir(DATA_ROOT),  f'missing {DATA_ROOT}'

## Cell 2 — Data loading & heatmap helpers

If your `merged_df.csv` uses different column names than `refined_label` / `frame_id`, edit `LABEL_COL` / `FRAME_COL` below.

In [ ]:
POINT_X, POINT_Y = 'x', 'y'
POINT_V   = 'velocity'
POINT_SNR = 'snr'
LABEL_COL = 'refined_label'
FRAME_COL = 'frame_id'


def load_all(data_root):
    files = glob.glob(os.path.join(data_root, '**', 'merged_df.csv'), recursive=True)
    if not files:
        raise FileNotFoundError(f'No merged_df.csv under {data_root}')
    dfs = []
    for f in files:
        d = pd.read_csv(f)
        d['__src'] = os.path.basename(os.path.dirname(f))
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)


def build_heatmap(points, K):
    hm = np.zeros((K, K, NUM_CHANNELS), dtype=np.float32)
    if points.empty:
        return hm
    xs = points[POINT_X].to_numpy()
    ys = points[POINT_Y].to_numpy()
    snr = points[POINT_SNR].to_numpy() if POINT_SNR in points.columns else np.ones(len(points))
    vel = points[POINT_V].to_numpy()   if POINT_V   in points.columns else np.zeros(len(points))
    xi = np.floor((xs - X_MIN) / (X_MAX - X_MIN) * K).astype(int)
    yi = np.floor((ys - Y_MIN) / (Y_MAX - Y_MIN) * K).astype(int)
    m = (xi >= 0) & (xi < K) & (yi >= 0) & (yi < K)
    for u, v, s, w in zip(xi[m], yi[m], snr[m], vel[m]):
        if s > hm[v, u, 0]:
            hm[v, u, 0] = s
        hm[v, u, 1] += w
    counts = (hm[..., 0] > 0).astype(np.float32)
    hm[..., 1] = np.divide(hm[..., 1], counts, out=np.zeros_like(hm[..., 1]), where=counts > 0)
    return hm


def make_sequences(df, L, K):
    X, y = [], []
    for src, g in df.groupby('__src', sort=False):
        frames = sorted(g[FRAME_COL].unique())
        per_frame_hm, per_frame_lbl = {}, {}
        for f in frames:
            sub = g[g[FRAME_COL] == f]
            per_frame_hm[f] = build_heatmap(sub, K)
            per_frame_lbl[f] = int(sub[LABEL_COL].mode().iloc[0])
        for i in range(L - 1, len(frames)):
            window = [per_frame_hm[frames[j]] for j in range(i - L + 1, i + 1)]
            X.append(np.stack(window, axis=0))
            y.append(per_frame_lbl[frames[i]])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int32)


df = load_all(DATA_ROOT)
print(f'Loaded {len(df):,} rows from {df["__src"].nunique()} experiments.')
print('Columns:', list(df.columns))
assert LABEL_COL in df.columns, f"expected '{LABEL_COL}'; got {list(df.columns)}"

## Cell 3 — Inference benchmark (Table 5)

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
n_params = int(model.count_params())
size_mb  = round(os.path.getsize(MODEL_PATH) / 1e6, 2)
print('Params:', n_params, '| Size MB:', size_mb)


def bench(model, device):
    with tf.device(device):
        x = tf.random.uniform((1, SEQ_LEN, GRID, GRID, NUM_CHANNELS), dtype=tf.float32)
        for _ in range(N_WARMUP):
            _ = model(x, training=False)
        times = []
        for _ in range(N_TRIALS):
            t0 = time.perf_counter()
            _ = model(x, training=False)
            times.append((time.perf_counter() - t0) * 1000.0)
    a = np.asarray(times)
    return {
        'mean_ms':   float(a.mean()),
        'median_ms': float(np.median(a)),
        'p95_ms':    float(np.percentile(a, 95)),
        'std_ms':    float(a.std()),
        'n_trials':  N_TRIALS,
    }


results = {
    'model_path':   MODEL_PATH,
    'n_parameters': n_params,
    'size_mb':      size_mb,
    'input_shape':  [1, SEQ_LEN, GRID, GRID, NUM_CHANNELS],
    'cpu':          bench(model, '/CPU:0'),
}
if tf.config.list_physical_devices('GPU'):
    results['gpu']      = bench(model, '/GPU:0')
    results['gpu_name'] = str(tf.config.list_physical_devices('GPU')[0].name)
else:
    results['gpu'] = None

with open('benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## Cell 4 — Ablation (Table 3)

Six variants: baseline, CNN-only, L∈{3,5,7}, K∈{32,48,64}. Writes `ablation_results.json` incrementally so even a partial run is useful.

In [ ]:
from tensorflow.keras.layers import (GRU, BatchNormalization, Conv2D, Dense,
                                     Dropout, Flatten, Input, Lambda,
                                     MaxPooling2D, TimeDistributed)


def build_full(L, K):
    cnn = tf.keras.Sequential([
        Conv2D(32, 3, activation='relu', padding='same', input_shape=(K, K, NUM_CHANNELS)),
        BatchNormalization(), MaxPooling2D(2),
        Conv2D(64,  3, activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2),
        Conv2D(128, 3, activation='relu', padding='same'), BatchNormalization(), MaxPooling2D(2),
        Flatten(),
    ])
    inp = Input(shape=(L, K, K, NUM_CHANNELS))
    x = TimeDistributed(cnn)(inp)
    x = GRU(128, dropout=0.3, recurrent_dropout=0.2)(x)
    x = Dropout(0.4)(x); x = Dense(64, activation='relu')(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inp, out)


def build_cnn_only(L, K):
    inp = Input(shape=(L, K, K, NUM_CHANNELS))
    last = Lambda(lambda t: t[:, -1])(inp)
    x = Conv2D(32, 3, activation='relu', padding='same')(last); x = BatchNormalization()(x); x = MaxPooling2D(2)(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x);    x = BatchNormalization()(x); x = MaxPooling2D(2)(x)
    x = Conv2D(128,3, activation='relu', padding='same')(x);    x = BatchNormalization()(x); x = MaxPooling2D(2)(x)
    x = Flatten()(x); x = Dropout(0.4)(x); x = Dense(64, activation='relu')(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inp, out)


def run_variant(name, build_fn, L, K, epochs):
    print(f'\n=== {name}  (L={L}, K={K}) ===', flush=True)
    t0 = time.time()
    X, y = make_sequences(df, L, K)
    print(f'  X={X.shape}  y={y.shape}  classes={np.unique(y)}', flush=True)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)
    cw = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
    cw = {int(c): float(w) for c, w in zip(np.unique(ytr), cw)}
    model = build_fn(L, K)
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    es = tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy')
    model.fit(Xtr, ytr, validation_split=0.1, epochs=epochs, batch_size=32,
              class_weight=cw, callbacks=[es], verbose=2)
    loss, acc = model.evaluate(Xte, yte, verbose=0)
    yp = np.argmax(model.predict(Xte, verbose=0), axis=1)
    rep = classification_report(yte, yp, output_dict=True, zero_division=0)
    return {
        'name': name, 'L': L, 'K': K,
        'test_accuracy': float(acc),
        'macro_f1':      float(rep['macro avg']['f1-score']),
        'weighted_f1':   float(rep['weighted avg']['f1-score']),
        'n_params': int(model.count_params()),
        'n_train':  int(len(Xtr)), 'n_test': int(len(Xte)),
        'wall_seconds': round(time.time() - t0, 1),
    }


variants = [
    ('full_L5_K48',     build_full,     5, 48),
    ('cnn_only_L5_K48', build_cnn_only, 5, 48),
    ('full_L3_K48',     build_full,     3, 48),
    ('full_L7_K48',     build_full,     7, 48),
    ('full_L5_K32',     build_full,     5, 32),
    ('full_L5_K64',     build_full,     5, 64),
]
out = []
for name, fn, L, K in variants:
    try:
        r = run_variant(name, fn, L, K, ABLATION_EPOCHS)
    except Exception as e:
        r = {'name': name, 'L': L, 'K': K, 'error': str(e)}
    out.append(r)
    with open('ablation_results.json', 'w') as f:
        json.dump(out, f, indent=2)
print('\nDone.\n' + json.dumps(out, indent=2))

## Cell 5 — Threshold sweep (Table 4)

Auto-finds any CSV under `results_outputs_tft/` that contains both `state_anomaly_score` and `duration_z_score` columns (produced by `03_tft_anomaly_v2.ipynb` Part 3).

In [ ]:
candidates = []
for f in glob.glob('results_outputs_tft/**/*.csv', recursive=True):
    try:
        cols = pd.read_csv(f, nrows=0).columns
        if 'state_anomaly_score' in cols and 'duration_z_score' in cols:
            candidates.append(f)
    except Exception:
        pass
print('Found', len(candidates), 'candidate CSVs:')
for c in candidates: print(' ', c)

TAU_S = [0.25, 0.5, 1.0, 2.0]
TAU_D = [0.5, 1.0, 2.0, 3.0]

summary = {}
for f in candidates:
    scen = os.path.basename(f).replace('.csv', '')
    df_s = pd.read_csv(f)
    seq = df_s['state_anomaly_score'].dropna()
    dur = df_s['duration_z_score'].dropna().abs()
    row = {'n_seq': int(len(seq)), 'n_dur': int(len(dur))}
    for t in TAU_S:
        row[f'seq_pct_gt_{t}'] = float((seq > t).mean() * 100) if len(seq) else None
    for t in TAU_D:
        row[f'dur_pct_gt_{t}'] = float((dur > t).mean() * 100) if len(dur) else None
    summary[scen] = row

with open('threshold_sweep.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## After running

Paste the three JSONs back: `benchmark_results.json`, `ablation_results.json`, `threshold_sweep.json`.